In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg, desc

In [2]:
spark = SparkSession.builder.appName("DataFrameJoining").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/19 19:34:55 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/19 19:34:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/19 19:35:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
customers_data = [
    (1, "Иван Петров", "ivan@example.com"),
    (2, "Анна Сидорова", "anna@example.com"),
    (3, "Елена Иванова", "elena@example.com"),
    (4, "Петр Смирнов", "petr@example.com"),
    (5, "Мария Кузнецова", "maria@example.com")
]

orders_data = [
    (101, 1, "2023-09-15", 999.99),
    (102, 2, "2023-09-16", 1199.00),
    (103, 3, "2023-09-17", 239.97),
    (104, 2, "2023-09-18", 699.99),
    (105, 4, "2023-09-19", 449.99)
]

products_data = [
    (201, "Ноутбук", 999.99, 10),
    (202, "Смартфон", 599.50, 20),
    (203, "Наушники", 79.99, 50),
    (204, "Планшет", 449.99, 15)
]

order_items_data = [
    (101, 201, 1),
    (102, 202, 2),
    (103, 203, 3),
    (104, 202, 1),
    (105, 204, 1)
]

In [5]:
customers_schema = ["customer_id", "customer_name", "email"]
customers_df = spark.createDataFrame(customers_data, customers_schema)

orders_schema = ["order_id", "customer_id", "order_date", "total_amount"]
orders_df = spark.createDataFrame(orders_data, orders_schema)

products_schema = ["product_id", "product_name", "price", "stock_quantity"]
products_df = spark.createDataFrame(products_data, products_schema)

order_items_schema = ["order_id", "product_id", "quantity"]
order_items_df = spark.createDataFrame(order_items_data, order_items_schema)

In [34]:
df_customer_orders = (
    customers_df.alias("c")
        .join(orders_df.alias("o"), "customer_id")
        .orderBy(col("order_date").desc())
)
df_customer_orders.show()

+-----------+-------------+-----------------+--------+----------+------------+
|customer_id|customer_name|            email|order_id|order_date|total_amount|
+-----------+-------------+-----------------+--------+----------+------------+
|          4| Петр Смирнов| petr@example.com|     105|2023-09-19|      449.99|
|          2|Анна Сидорова| anna@example.com|     104|2023-09-18|      699.99|
|          3|Елена Иванова|elena@example.com|     103|2023-09-17|      239.97|
|          2|Анна Сидорова| anna@example.com|     102|2023-09-16|      1199.0|
|          1|  Иван Петров| ivan@example.com|     101|2023-09-15|      999.99|
+-----------+-------------+-----------------+--------+----------+------------+



In [27]:
from pyspark.sql.functions import col, round

In [19]:
df_order_details = (
    orders_df.alias("o")
        .join(order_items_df.alias("oi"), "order_id")
        .join(products_df.alias("p"), "product_id")
        .filter((col("quantity") * col("price")) >  500)
)
df_order_details.show()

+----------+--------+-----------+----------+------------+--------+------------+------+--------------+
|product_id|order_id|customer_id|order_date|total_amount|quantity|product_name| price|stock_quantity|
+----------+--------+-----------+----------+------------+--------+------------+------+--------------+
|       201|     101|          1|2023-09-15|      999.99|       1|     Ноутбук|999.99|            10|
|       202|     104|          2|2023-09-18|      699.99|       1|    Смартфон| 599.5|            20|
|       202|     102|          2|2023-09-16|      1199.0|       2|    Смартфон| 599.5|            20|
+----------+--------+-----------+----------+------------+--------+------------+------+--------------+



In [20]:
df_all_customers = (
    customers_df
        .join(orders_df, "customer_id", how="left")
)
df_all_customers.show()

+-----------+---------------+-----------------+--------+----------+------------+
|customer_id|  customer_name|            email|order_id|order_date|total_amount|
+-----------+---------------+-----------------+--------+----------+------------+
|          1|    Иван Петров| ivan@example.com|     101|2023-09-15|      999.99|
|          2|  Анна Сидорова| anna@example.com|     104|2023-09-18|      699.99|
|          2|  Анна Сидорова| anna@example.com|     102|2023-09-16|      1199.0|
|          3|  Елена Иванова|elena@example.com|     103|2023-09-17|      239.97|
|          5|Мария Кузнецова|maria@example.com|    NULL|      NULL|        NULL|
|          4|   Петр Смирнов| petr@example.com|     105|2023-09-19|      449.99|
+-----------+---------------+-----------------+--------+----------+------------+



In [29]:
df_customer_total = (
    customers_df
        .join(orders_df, "customer_id")
        .join(order_items_df, "order_id")
        .join(products_df, "product_id")
        .groupBy("customer_id")
        .agg(round(sum(col("quantity") * col("price")), 2).alias("total_purchases"))
        .sort(col("total_purchases").desc())
)
df_customer_total.show()

+-----------+---------------+
|customer_id|total_purchases|
+-----------+---------------+
|          2|         1798.5|
|          1|         999.99|
|          4|         449.99|
|          3|         239.97|
+-----------+---------------+



In [33]:
df_top_products = (
    products_df
        .join(order_items_df, "product_id")
        .groupBy("product_id")
        .agg(sum(col("quantity")).alias("total_quantity"))
        .orderBy(col("total_quantity").desc(), col("product_id").desc())
        .limit(3)
)
df_top_products.show()

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|       203|             3|
|       202|             3|
|       204|             1|
+----------+--------------+

